# Train PoseRFM

PoseRFM, the Riemannian flow matching prior of PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026), on SO(3)^19 (`hp/models/poserfm.py`):
- **data:** by default every 10th frame of our local AMASS (`~/data/AMASS_rotations_c10`), the official train / valid split (`ORIGINAL_AMASS_SPLITS`): the data of the best RSGM so far (trial 34 of round 1: 139.1-139.2 nats on the test split, `docs/rsgm_mirror_augmentation.md`), so the two can be compared;
- **model and training:** `config/models/poserfm.yaml`, the settings of the paper: MLP 4 x 512, silu, AdamW lr 1e-3 with weight decay 1e-4, EMA 0.99, batch 4096, 50k steps. One deviation: the paper lowers the learning rate on plateaus (ReduceLROnPlateau), which the trainer does not have yet; here the step schedule halves it once, at 70% of the steps;
- **device:** mps.

The validation loss is the flow matching loss of the EMA weights on the valid split, with fixed times and source points, so validations are comparable with each other. It is no likelihood (the likelihood is in the quick check at the end, and in the evaluation notebooks). Compare runs also with `explore/evaluation/training_curves.ipynb`.

A finished run is not trained again; `RETRAIN = True` trains a new one (runs are never overwritten).

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.evaluation.data import load_reference
from hp.models.io import load_model
from hp.training import cli
from hp.training.config import load_config

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_c10"     # every 10th frame: the data of the best RSGM (trial 34)
#ROTATIONS = "~/data/AMASS_rotations_c30"    # every 30th frame
#ROTATIONS = "~/data/AMASS_rotations"        # all frames (needs "data.load_dtype=float32" in OVERRIDES)
SPLIT = "original"                           # the official AMASS split
RUN_ROOT = "~/data/experiments"              # runs go to RUN_ROOT/RUN_NAME/run_0001, run_0002, ...
RUN_NAME = "poserfm_c10"
DEVICE = "mps"                               # "cpu", "mps" or "cuda:0"
SEED = 10
RETRAIN = False                              # True: train a new run even if a finished one exists
OVERRIDES = []                               # changes to config/models/poserfm.yaml, e.g. ["train.max_iterations=20000"]
N_CHECK = 500                                # valid poses for the quick likelihood check at the end
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
argv = ["model=poserfm", f"run_root={RUN_ROOT}", f"run_name={RUN_NAME}", f"seed={SEED}", f"data.rotations={ROTATIONS}",
        f"data.split={SPLIT}", f"train.device={DEVICE}", *OVERRIDES]
cfg = load_config(argv)
print(OmegaConf.to_yaml(cfg))

train_frames = sum(torch.load(expand(ROTATIONS) / f"{d}.pt", mmap=True).shape[0] for d in ORIGINAL_AMASS_SPLITS["train"])
iterations, batch = cfg.train.max_iterations, cfg.train.batch_size
print(f"{iterations:,} iterations x batch {batch} = {iterations * batch / train_frames:.1f} epochs over {train_frames:,} training frames")

## Train

With `scripts/train_model.py` (`hp.training.cli.main`). The wall time is stored in `train_time.json` of the run folder, for the comparison of the training budgets.

In [ ]:
def finished_run(root):
    # The newest run folder below root that finished (status.json says so), or None.
    runs = sorted(p for p in root.glob("run_[0-9][0-9][0-9][0-9]")
                  if (p / "status.json").is_file() and json.loads((p / "status.json").read_text())["finished"])
    return runs[-1] if runs else None


run_dir = None if RETRAIN else finished_run(expand(RUN_ROOT) / RUN_NAME)
if run_dir is not None:
    print(f"already trained: {run_dir}")
else:
    start = time.perf_counter()
    run_dir = cli.main(argv)
    seconds = time.perf_counter() - start
    (run_dir / "train_time.json").write_text(json.dumps({"seconds": seconds, "device": DEVICE}))
    print(f"trained in {seconds / 60:.1f} min")

## Loss curves

Training loss of every step (and its moving average) and the validation loss. Epochs over the training frames on the top axis.

In [ ]:
SMOOTH = 200  # steps of the moving average

history = json.loads((run_dir / "history.json").read_text())
loss = np.asarray(history["loss"], dtype=float)
steps = np.arange(1, len(loss) + 1)
smooth = np.convolve(loss, np.ones(SMOOTH) / SMOOTH, mode="valid")
per_epoch = train_frames / batch  # steps per epoch

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, loss, color="C0", alpha=0.15, lw=0.5)
ax.plot(steps[SMOOTH - 1:], smooth, color="C0", label=f"training (moving average of {SMOOTH})")
ax.plot(history["valid_iteration"], history["valid_loss"], "o-", color="C1", label="validation (EMA weights, fixed noise)")
ax.set_ylim(np.percentile(loss, 1) * 0.95, np.percentile(loss, 99.5) * 1.05)
ax.set_xlabel("step")
ax.set_ylabel("flow matching loss")
ax.secondary_xaxis("top", functions=(lambda s: s / per_epoch, lambda e: e * per_epoch)).set_xlabel("epoch")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

if history["valid_loss"]:
    best = int(np.argmin(history["valid_loss"]))
    print(f"validation loss: last {history['valid_loss'][-1]:.3f}, best {history['valid_loss'][best]:.3f} at step {history['valid_iteration'][best]:,}")

## Quick check

The EMA weights: samples must be valid rotations, and the log-likelihood of `N_CHECK` poses of the valid split (the ODE backwards with the exact divergence) must be finite. Sampling and likelihood times are printed for the comparison of the costs. The full evaluation is in `explore/evaluation/rsgm_huproso_comparison.ipynb` and `explore/evaluation/rsgm_performance_tests.ipynb`.

In [ ]:
model = load_model(run_dir, device=DEVICE, weights="weights_ema")
torch.manual_seed(SEED)
start = time.perf_counter()
samples = model.sample(1000).cpu().double()
sample_seconds = time.perf_counter() - start
eye = torch.eye(3, dtype=torch.float64)
print(f"samples: {tuple(samples.shape)}, max |R^T R - I| = {(samples.transpose(-1, -2) @ samples - eye).abs().max():.1e}, "
      f"min det = {torch.linalg.det(samples).min():.4f}, {sample_seconds:.2f} s per 1000 samples")

valid = load_reference("valid", ROTATIONS, SPLIT)
poses = valid[torch.randperm(len(valid), generator=torch.Generator().manual_seed(SEED))[:N_CHECK]]
start = time.perf_counter()
with torch.no_grad():
    ll = torch.cat([model.log_prob(poses[i:i + 250].to(model.device)).cpu() for i in range(0, len(poses), 250)])
print(f"log-likelihood of {len(poses)} valid poses: {ll.mean():.2f} +- {ll.std() / len(ll) ** 0.5:.2f} (s.e.m.), "
      f"{(~torch.isfinite(ll)).sum()} non-finite, {(time.perf_counter() - start) / len(poses) * 1000:.1f} ms per pose")